# Example: Bandit Elasticity Learning

In this example, we apply the **epsilon-greedy multi-armed bandit** to learn the best CES elasticity $\eta$ per sentiment regime. Instead of using the Session 2 heuristic $\eta(\lambda_t) = \eta_{\min} + (\eta_{\max} - \eta_{\min})/(1 + |\lambda_t|)$, we let the bandit discover which $\eta$ value produces the highest utility in bearish, neutral, and bullish conditions.

The Session 2 heuristic baked the elasticity-vs-sentiment relationship into a hand-designed formula: $\eta$ falls smoothly with $|\lambda_t|$ and is symmetric in bull and bear. This example removes that prior and lets a bandit pick the best elasticity per regime from reward feedback alone, and asks: did the heuristic miss something a learner can find, or was the formula already close enough that the learning effort does not pay?

> __Learning Objectives:__
>
> By the end of this example, you will be able to:
> * __Explain regime-specific elasticity learning:__ Describe how a per-regime bandit can discover elasticity values that a fixed sentiment-symmetric formula cannot adapt to. Read per-regime arm means and the reward trace to interpret what the bandit learned and how confident it is.
> * __Interpret distributional evidence on elasticity selection:__ Compare bandit-learned, heuristic, and fixed-elasticity engines on the same path ensemble with the same bias treatment, so distributional differences in median wealth, drawdown, and risk-adjusted growth are attributable to the elasticity-selection method itself.
> * __Assess whether learning beats the heuristic:__ Use the win rate and median paired-excess between the bandit and the heuristic to judge whether per-regime learning produces a real edge. Read the paired distribution to separate genuine improvement from sampling noise on a fixed-seed ensemble.

Let's see if a learner can beat a heuristic.

___

## Setup, Data and Prerequisites
We begin by loading our packages and helper functions via [the `Include.jl` file](./Include.jl). This activates the local [Julia](https://julialang.org) environment and loads all dependencies.

In [ ]:
# --- Load packages and helper functions ---
include("Include.jl"); # The Include.jl file activates the local Julia environment and imports all dependencies.

### Constants
In the section below, we set some constants that will be used throughout the notebook. We can modify these constants to explore different scenarios and allocations.

See the comments in the code for more details on each constant, its purpose, units, etc.

In [ ]:
# Eta-bandit configuration
B₀ = 10_000.0                                # starting budget (USD)
Δt = 1.0 / 252.0                             # trading-day step (years)
L_short = 21                                 # short EMA window (days) for the sentiment crossover
L_long = 63                                  # long EMA window (days) for the sentiment crossover
L_growth = 10                                # EMA window (days) for smoothed market growth rate
GAIN = 10.0                                  # gain constant G for the λ sentiment signal (dimensionless)
offset = L_short + L_long                    # warmup offset before trading begins (days)
SCENARIO_SEED = 2026                         # RNG seed for the hybrid-SIM scenario (locked across Examples 1-2)
ETA_GRID = [0.5, 1.0, 1.5, 2.0, 3.0, 5.0]    # discrete CES elasticity arms (one bandit arm per η value)
BANDIT_ITERS_SINGLE = 500                    # bandit iteration budget on the single-path Task 1 run
BANDIT_ALPHA = 0.1                           # constant learning rate for bandit arm-mean updates
LAMBDA_THRESHOLD = 0.5                       # |λ| boundary partitioning sentiment into bear/neutral/bull bins
TRIGGER_MAX_DRAWDOWN = 0.15                  # drawdown trigger threshold (de-risks to cash, fraction)
TRIGGER_MAX_TURNOVER = 0.50                  # max fraction of wealth traded per rebalance
ETA_BOUNDS = (0.5, 5.0)                      # adaptive-η bounds for the heuristic baseline (η_min, η_max)
ALLOCATION_EPSILON = 0.1                     # ε floor for non-preferred assets in the Cobb-Douglas allocator
FIXED_ETA = 2.0                              # elasticity used by the Fixed-η baseline arm in the Task 2 scorecard
ENGINE_PRIOR_CCGR_PCT = 8.0                  # bias-correction anchor (% /yr); matches Session 2 + EWLS notebook

Load the same Session 1 artifacts and regenerate the same forward path ensemble used in Example 1 (EWLS Engine Replay). The deterministic seed ensures identical paths.

In the code block below, we load the Session 1 minimum-variance pack, build the SIM-parameter dictionary keyed by ticker, instantiate the [`MyMarketSurrogateModel`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session1/#eCornellAIFinance.MyMarketSurrogateModel) and [`MyPortfolioSurrogateModel`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session1/#eCornellAIFinance.MyPortfolioSurrogateModel) calibrated in Session 1, and call [`generate_hybrid_scenario(...)`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session3/#eCornellAIFinance.generate_hybrid_scenario) with the locked seed to produce the forward path ensemble.

The cell returns:

* `my_tickers::Vector{String}`: Session 1 ticker universe (same order as `allocation_weights`).
* `sim_estimates::Vector{MySIMParameterEstimate}`: per-ticker SIM fits with α, β, σ_ε.
* `sim_params::Dict{String,Tuple{Float64,Float64,Float64}}`: allocator-adapter view of `sim_estimates`, keyed by ticker.
* `g_f::Float64`: continuously compounded annual risk-free rate (1/yr); used to discount terminal wealth into NPV against the risk-free baseline.
* [`market_model::MyMarketSurrogateModel`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session1/#eCornellAIFinance.MyMarketSurrogateModel): 50-state generative model market surrogate fit to SPY.
* [`portfolio::MyPortfolioSurrogateModel`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session1/#eCornellAIFinance.MyPortfolioSurrogateModel): per-ticker generative marginals and Student-t copula for path composition.
* `calib::Dict{String,Any}`: full SIM calibration dict (source of truth for α, β, σ_ε, and bootstrap SEs).
* `start_prices::Dict{String,Float64}`: per-ticker starting prices for forward scenarios.
* `N::Int`: number of tickers in the universe.
* `scenario::MyBacktestScenario`: hybrid forward path ensemble (seed-locked for reproducibility across Examples 1-2).

In [ ]:
(; my_tickers, sim_estimates, sim_params, g_f,
   market_model, portfolio, calib, start_prices, N, scenario) = let
    # --- Step 1: Load S1 artifacts ---
    minvar = load_results(joinpath(_PATH_TO_DATA_S1, "minvar-allocation.jld2"));
    my_tickers    = minvar["my_tickers"]::Vector{String};
    sim_estimates = minvar["sim_estimates"];
    g_f           = haskey(minvar, "g_f") ? Float64(minvar["g_f"]) : Float64(minvar["r_f"]);
    sim_params = Dict{String,Tuple{Float64,Float64,Float64}}(
        e.ticker => (e.α, e.β, e.σ_ε) for e in sim_estimates
    );

    # --- Step 2: Surrogates and scenario ---
    market_model = MyMarketSurrogateModel();
    portfolio    = MyPortfolioSurrogateModel();
    calib        = load_results(joinpath(_PATH_TO_DATA_S1, "sim-parameter-estimates.jld2"));
    snap        = MyCurrentPrices();
    snap_lookup = Dict(snap["tickers"] .=> snap["prices"]);
    start_prices = Dict(t => snap_lookup[t] for t in my_tickers);

    # --- Step 3: Dimensions ---
    N         = length(my_tickers);
    n_trading_days   = 252;
    T_total          = offset + n_trading_days;

    # --- Step 4: Regenerate scenario (same seed as Example 1) ---
    Random.seed!(SCENARIO_SEED);
    scenario = generate_hybrid_scenario(
        market_model, portfolio, calib, my_tickers;
        n_paths = 500, n_steps = T_total,
        start_prices = start_prices, label = "S3 Eta-Bandit", seed = SCENARIO_SEED);

    println("Loaded $(N) tickers, generated $(scenario.n_paths)-path scenario")
    (my_tickers = my_tickers, sim_estimates = sim_estimates, sim_params = sim_params,
     g_f = g_f, market_model = market_model, portfolio = portfolio,
     calib = calib, start_prices = start_prices, N = N, scenario = scenario)
end

___
## Task 1: Train the Elasticity Bandit on a Single Path
In this task, we extract the first path from the scenario, build the rebalancing context, and run the elasticity bandit with 6 arms ($\eta \in \{0.5, 1.0, 1.5, 2.0, 3.0, 5.0\}$) and a regime threshold $\theta = 0.5$. The bandit learns independently in bearish, neutral, and bullish bins, calling [`solve_eta_bandit(...)`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session3/#eCornellAIFinance.solve_eta_bandit) on a [`MyEtaBanditModel`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session3/#eCornellAIFinance.MyEtaBanditModel) instance.

> __What should we see?__
>
> The reward trace should show initial exploration (noisy) converging to a stable level as the bandit discovers the best arm per regime. The per-regime bar chart should reveal whether different regimes prefer different elasticity values.

The code block below returns `bandit_result::`[`MyEtaBanditResult`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session3/#eCornellAIFinance.MyEtaBanditResult) carrying the per-regime arm means, pull counts, reward trace, and best-elasticity map.

In [ ]:
bandit_result = let
    # --- Step 1: Extract first path ---
    p_idx = 1;
    T_total = scenario.n_steps;
    mkt = scenario.market_paths[p_idx, :];
    pmatrix = zeros(T_total, N + 1);
    pmatrix[:, 1] = 1:T_total;
    for k in 1:N
        pmatrix[:, k + 1] = scenario.price_paths[p_idx, :, k];
    end

    # --- Step 2: Compute lambda series ---
    ema_s = compute_ema(mkt; window = L_short);
    ema_l = compute_ema(mkt; window = L_long);
    lambda_series = compute_lambda(ema_s, ema_l; G = GAIN);
    lambda_series[1:offset] .= 0.0;
    gm_raw = compute_market_growth(mkt; Δt = Δt);
    gm_ema = compute_ema(gm_raw; window = L_growth);

    # --- Step 3: Build context ---
    ctx = build(MyRebalancingContextModel, (
        B = B₀, tickers = my_tickers, marketdata = pmatrix,
        marketfactor = gm_ema, sim_parameters = sim_params,
        lambda = 0.0, Δt = Δt, epsilon = ALLOCATION_EPSILON
    ));

    # --- Step 4: Build and run eta-bandit ---
    η_grid = ETA_GRID;
    bandit = build(MyEtaBanditModel, (
        eta_grid = η_grid, n_iterations = BANDIT_ITERS_SINGLE, alpha = BANDIT_ALPHA, lambda_threshold = LAMBDA_THRESHOLD
    ));

    time_indices = collect((offset + 1):T_total);
    bandit_result = solve_eta_bandit(bandit, ctx, lambda_series, time_indices);

    # --- Step 5: Build per-regime results DataFrame (rows = regime, columns = best η, total pulls, per-arm μ̂) ---
    regime_list = [:bearish, :neutral, :bullish];
    regime_df = DataFrame(
        "Regime"      => string.(regime_list),
        "Best η"      => [bandit_result.best_eta_per_regime[r] for r in regime_list],
        "Total pulls" => [sum(bandit_result.arm_counts_per_regime[r]) for r in regime_list],
    );
    for (i, eta_val) in enumerate(η_grid)
        regime_df[!, "μ̂(η=$(eta_val))"] = [round(bandit_result.arm_means_per_regime[r][i], digits = 3) for r in regime_list];
    end
    println("Eta-bandit results (Path $(p_idx)):")
    pretty_table(regime_df; backend = :text,
        fit_table_in_display_horizontally = false,
        fit_table_in_display_vertically = false,
        table_format = TextTableFormat(borders = text_table_borders__compact))

    # --- Panel 1: Reward trace ---
    p1 = plot(bandit_result.reward_history, label="Reward", alpha=0.3, color=:gray50,
        ylabel="CES Utility", title="Eta-Bandit: Reward Trace", size=(800, 250))

    # --- Panel 2: Exploration decay ---
    p2 = plot(bandit_result.exploration_history, label="ε_t", color=:coral, linewidth=2,
        ylabel="ε", title="Exploration Decay", xlabel="Iteration", size=(800, 200))

    display(plot(p1, p2, layout=grid(2,1, heights=[0.55, 0.45]), size=(800, 450)))

    # --- Panel 3: Per-regime arm means ---
    p3 = plot(layout=(1,3), size=(900, 300), title=["Bearish" "Neutral" "Bullish"])
    for (j, regime) in enumerate([:bearish, :neutral, :bullish])
        bar!(p3[j], string.(η_grid), bandit_result.arm_means_per_regime[regime],
            label="", color=:steelblue, xlabel="η", ylabel=j==1 ? "Mean Reward" : "")
    end
    display(p3)
    bandit_result
end

___
## Task 2: Backtest Bandit-Learned Elasticity Across All Paths
In this task, we take the elasticity map learned on path 1 and apply it across the entire path ensemble via [the `backtest_eta_bandit(...)` function](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session3/#eCornellAIFinance.backtest_eta_bandit). We also run the heuristic $\eta(\lambda)$ engine and a fixed-$\eta$ engine on the **same scenario** as inline baselines so all three columns share generating conditions and bias treatment.

> __What should we see?__
>
> All three arms are daily-rebalanced and therefore inherit the SIM rebalancing-alpha artifact that Session 2 documented; we apply the same `apply_bias_correction` anchor (`ENGINE_PRIOR_CCGR_PCT`) to all three so the median levels are comparable and the relative ordering reflects the elasticity-selection method, not the artifact. The bandit-learned elasticity should match or beat the heuristic on median metrics; the fixed-elasticity baseline gives the no-regime-awareness reference point.

The code block below returns `bandit_bt::`[`MyBacktestResult`](https://varnerlab.org/eCornell-AI-finance-lectures/dev/session3/#eCornellAIFinance.MyBacktestResult), the bias-corrected per-path terminal-wealth, drawdown, and Sharpe arrays for the bandit-learned-elasticity engine across the path ensemble.

In [ ]:
bandit_bt = let
    # --- Step 0: Local bias-correction helper (mirrors the EWLS notebook) ---
    function _apply_bias(r::MyBacktestResult, bias_pct_per_yr::Float64, Δt::Float64)::MyBacktestResult
        n_t, n_p = size(r.wealth_paths);
        drag = exp.(-bias_pct_per_yr / 100.0 .* (0:(n_t-1)) .* Δt);
        Wc = r.wealth_paths .* drag;
        final_wealth_c  = Wc[end, :];
        max_drawdowns_c = zeros(n_p);
        sharpe_ratios_c = zeros(n_p);
        for p ∈ 1:n_p
            wealth = Wc[:, p];
            peak = accumulate(max, wealth);
            max_drawdowns_c[p] = maximum((peak .- wealth) ./ peak);
            vol = std(diff(wealth) ./ wealth[1:end-1]) * sqrt(252);
            mean_ret = (wealth[end] / wealth[1] - 1.0);
            sharpe_ratios_c[p] = vol > 1e-6 ? mean_ret / vol : 0.0;
        end
        rc = MyBacktestResult();
        rc.scenario_label = r.scenario_label;
        rc.strategy_label = r.strategy_label * " [bias-corrected $(round(bias_pct_per_yr, digits=2))pp/yr]";
        rc.final_wealth   = final_wealth_c;
        rc.max_drawdowns  = max_drawdowns_c;
        rc.sharpe_ratios  = sharpe_ratios_c;
        rc.wealth_paths   = Wc;
        return rc;
    end

    # --- Step 1: Run bandit-learned eta engine via backtest_eta_bandit ---
    eta_map = bandit_result.best_eta_per_regime;
    bandit_bt_raw = backtest_eta_bandit(scenario, my_tickers, sim_params, eta_map;
        B₀ = B₀, offset = offset, L_short = L_short, L_long = L_long,
        GAIN = GAIN, L_growth = L_growth, lambda_threshold = LAMBDA_THRESHOLD);
    bandit_bt_raw.strategy_label = "Bandit-η raw";

    # --- Step 2: Run heuristic-η and Fixed-η engines inline on the same scenario, storing wealth paths ---
    n_trading = scenario.n_steps - offset;
    n_paths   = scenario.n_paths;
    heur_final_wealth  = zeros(n_paths);
    heur_max_dd        = zeros(n_paths);
    heur_sharpe        = zeros(n_paths);
    heur_wealth_paths  = zeros(n_trading + 1, n_paths);
    fixed_final_wealth = zeros(n_paths);
    fixed_max_dd       = zeros(n_paths);
    fixed_sharpe       = zeros(n_paths);
    fixed_wealth_paths = zeros(n_trading + 1, n_paths);

    for p in 1:n_paths
        mkt = scenario.market_paths[p, :];
        ema_s = compute_ema(mkt; window = L_short);
        ema_l = compute_ema(mkt; window = L_long);
        λ_p = compute_lambda(ema_s, ema_l; G = GAIN);
        λ_p[1:offset] .= 0.0;
        gm_raw = compute_market_growth(mkt; Δt = Δt);
        gm_e = compute_ema(gm_raw; window = L_growth);

        pmatrix = zeros(scenario.n_steps, N + 1);
        pmatrix[:, 1] = 1:scenario.n_steps;
        for k in 1:N
            pmatrix[:, k + 1] = scenario.price_paths[p, :, k];
        end

        ctx = build(MyRebalancingContextModel, (
            B = B₀, tickers = my_tickers, marketdata = pmatrix,
            marketfactor = gm_e, sim_parameters = sim_params,
            lambda = 0.0, Δt = Δt, epsilon = ALLOCATION_EPSILON
        ));
        rules = build(MyTriggerRules, (
            max_drawdown = TRIGGER_MAX_DRAWDOWN, max_turnover = TRIGGER_MAX_TURNOVER,
            rebalance_schedule = ones(Int, n_trading)
        ));

        # Heuristic-η: adaptive_eta = true with eta_bounds
        results_h = run_rebalancing_engine(ctx, rules, λ_p;
            offset = offset, allocator = :ces, adaptive_eta = true, eta_bounds = ETA_BOUNDS);
        w_h = compute_wealth_series(results_h, pmatrix, my_tickers; offset = offset);
        heur_wealth_paths[:, p] .= w_h;
        heur_final_wealth[p] = w_h[end];
        ret_h = diff(w_h) ./ w_h[1:end-1];
        pk_h = accumulate(max, w_h);
        heur_max_dd[p] = maximum((pk_h .- w_h) ./ pk_h);
        vol_h = std(ret_h) * sqrt(252);
        heur_sharpe[p] = vol_h > 0 ? (w_h[end]/w_h[1] - 1.0) / vol_h : 0.0;

        # Fixed-η: adaptive_eta = false with eta = FIXED_ETA
        results_f = run_rebalancing_engine(ctx, rules, λ_p;
            offset = offset, allocator = :ces, adaptive_eta = false, eta = FIXED_ETA);
        w_f = compute_wealth_series(results_f, pmatrix, my_tickers; offset = offset);
        fixed_wealth_paths[:, p] .= w_f;
        fixed_final_wealth[p] = w_f[end];
        ret_f = diff(w_f) ./ w_f[1:end-1];
        pk_f = accumulate(max, w_f);
        fixed_max_dd[p] = maximum((pk_f .- w_f) ./ pk_f);
        vol_f = std(ret_f) * sqrt(252);
        fixed_sharpe[p] = vol_f > 0 ? (w_f[end]/w_f[1] - 1.0) / vol_f : 0.0;
    end

    heur_bt_raw = MyBacktestResult();
    heur_bt_raw.scenario_label = scenario.label;
    heur_bt_raw.strategy_label = "Heuristic-η raw";
    heur_bt_raw.final_wealth   = heur_final_wealth;
    heur_bt_raw.max_drawdowns  = heur_max_dd;
    heur_bt_raw.sharpe_ratios  = heur_sharpe;
    heur_bt_raw.wealth_paths   = heur_wealth_paths;

    fixed_bt_raw = MyBacktestResult();
    fixed_bt_raw.scenario_label = scenario.label;
    fixed_bt_raw.strategy_label = "Fixed η=$(FIXED_ETA) raw";
    fixed_bt_raw.final_wealth   = fixed_final_wealth;
    fixed_bt_raw.max_drawdowns  = fixed_max_dd;
    fixed_bt_raw.sharpe_ratios  = fixed_sharpe;
    fixed_bt_raw.wealth_paths   = fixed_wealth_paths;

    # --- Step 3: Bias correction anchored to ENGINE_PRIOR_CCGR_PCT (same anchor as EWLS notebook) ---
    bandit_raw_med_ccgr = let
        gs = Float64[];
        for p ∈ 1:n_paths
            w = bandit_bt_raw.wealth_paths[:, p];
            push!(gs, log(w[end] / w[1]) / ((length(w) - 1) * Δt) * 100);
        end
        median(gs)
    end;
    bias_pct_per_yr = bandit_raw_med_ccgr - ENGINE_PRIOR_CCGR_PCT;
    println("  Bandit-η raw median CCGR: $(round(bandit_raw_med_ccgr, digits=2)) %/yr");
    println("  ENGINE_PRIOR_CCGR_PCT:    $(ENGINE_PRIOR_CCGR_PCT) %/yr");
    println("  Bias correction drag:     $(round(bias_pct_per_yr, digits=2)) pp/yr (applied to all three daily-rebalanced arms)");

    bandit_bt_c = _apply_bias(bandit_bt_raw, bias_pct_per_yr, Δt);
    heur_bt_c   = _apply_bias(heur_bt_raw,   bias_pct_per_yr, Δt);
    fixed_bt_c  = _apply_bias(fixed_bt_raw,  bias_pct_per_yr, Δt);

    # --- Step 4: Per-path NPV against the continuously-compounded risk-free baseline ---
    # NPV_p = W_T,p · exp(-g_f · T_active) − B₀.  Positive ⇒ path beat risk-free in today's dollars.
    T_active   = (scenario.n_steps - offset) * Δt;
    discount   = exp(-g_f * T_active);
    bandit_npv = bandit_bt_c.final_wealth .* discount .- B₀;
    heur_npv   = heur_bt_c.final_wealth   .* discount .- B₀;
    fixed_npv  = fixed_bt_c.final_wealth  .* discount .- B₀;

    # --- Step 5: Build the distributional comparison DataFrame (all three arms, full metrics) ---
    scorecard_df = DataFrame(
        "Metric"           => ["Median W/W₀", "Median Max DD (%)", "Median Sharpe",
                               "Median NPV (\$)", "Median NPV (% of B₀)", "P[NPV<0] (%)"],
        "Bandit-η"         => Any[
            round(median(bandit_bt_c.final_wealth) / B₀, digits = 3),
            round(median(bandit_bt_c.max_drawdowns) * 100, digits = 1),
            round(median(bandit_bt_c.sharpe_ratios), digits = 3),
            round(median(bandit_npv),              digits = 0),
            round(median(bandit_npv) / B₀ * 100,   digits = 2),
            round(mean(bandit_npv .< 0) * 100,     digits = 1),
        ],
        "Heuristic-η"      => Any[
            round(median(heur_bt_c.final_wealth) / B₀, digits = 3),
            round(median(heur_bt_c.max_drawdowns) * 100, digits = 1),
            round(median(heur_bt_c.sharpe_ratios), digits = 3),
            round(median(heur_npv),              digits = 0),
            round(median(heur_npv) / B₀ * 100,   digits = 2),
            round(mean(heur_npv .< 0) * 100,     digits = 1),
        ],
        "Fixed η=$(FIXED_ETA)" => Any[
            round(median(fixed_bt_c.final_wealth) / B₀, digits = 3),
            round(median(fixed_bt_c.max_drawdowns) * 100, digits = 1),
            round(median(fixed_bt_c.sharpe_ratios), digits = 3),
            round(median(fixed_npv),              digits = 0),
            round(median(fixed_npv) / B₀ * 100,   digits = 2),
            round(mean(fixed_npv .< 0) * 100,     digits = 1),
        ],
    );
    println("CES η strategies: distributional comparison ($(n_paths) paths, bias-corrected to anchor=$(ENGINE_PRIOR_CCGR_PCT) %/yr)")
    pretty_table(scorecard_df; backend = :text,
        fit_table_in_display_horizontally = false,
        fit_table_in_display_vertically = false,
        table_format = TextTableFormat(borders = text_table_borders__compact))

    # --- Step 6: Save for Example 3 (NPV arrays go through the hand-off) ---
    save_results(joinpath(_PATH_TO_DATA, "eta-bandit-results.jld2"), Dict(
        "bandit_final_wealth" => bandit_bt_c.final_wealth,
        "bandit_max_dd"       => bandit_bt_c.max_drawdowns,
        "bandit_sharpe"       => bandit_bt_c.sharpe_ratios,
        "bandit_npv"          => bandit_npv,
        "heur_final_wealth"   => heur_bt_c.final_wealth,
        "heur_max_dd"         => heur_bt_c.max_drawdowns,
        "heur_sharpe"         => heur_bt_c.sharpe_ratios,
        "heur_npv"            => heur_npv,
        "fixed_final_wealth"  => fixed_bt_c.final_wealth,
        "fixed_max_dd"        => fixed_bt_c.max_drawdowns,
        "fixed_sharpe"        => fixed_bt_c.sharpe_ratios,
        "fixed_npv"           => fixed_npv,
        "best_eta_per_regime" => bandit_result.best_eta_per_regime,
        "fixed_eta"           => FIXED_ETA,
        "bias_pct_per_yr"     => bias_pct_per_yr,
        "g_f"                 => g_f,
        "T_active"            => T_active,
    ));
    println("Saved to eta-bandit-results.jld2")
    bandit_bt_c
end

___
## Task 3: Bandit-Learned Elasticity vs. Heuristic Elasticity: Head-to-Head
In this task, we compute the per-path excess wealth (bandit minus heuristic) and visualize the distribution. This paired comparison controls for path-level variation and isolates the effect of the elasticity selection method.

> __What should we see?__
>
> If the bandit discovered genuinely better elasticity values, the excess distribution should be right-shifted (positive median). If the heuristic was already near-optimal, the distribution should be centered near zero.

In the code block below, we load the saved bandit and heuristic terminal-wealth arrays, compute the paired excess and win rate, and plot the histogram with a dashed zero line and a coral median marker.

In [ ]:
let
    # --- Step 1: Load bandit and heuristic results ---
    data = load_results(joinpath(_PATH_TO_DATA, "eta-bandit-results.jld2"));
    bandit_fw = data["bandit_final_wealth"];
    heur_fw   = data["heur_final_wealth"];

    # --- Step 2: Paired excess ---
    excess = bandit_fw .- heur_fw;
    win_rate = mean(excess .> 0);

    # --- Step 3: Paired-excess summary table ---
    excess_df = DataFrame(
        "Metric"     => ["Win rate (%)", "Median excess (\$)", "Median excess (% of W₀)", "Mean excess (\$)", "Mean excess (% of W₀)"],
        "Value"      => [
            round(win_rate * 100, digits = 1),
            round(median(excess), digits = 0),
            round(median(excess) / B₀ * 100, digits = 2),
            round(mean(excess), digits = 0),
            round(mean(excess) / B₀ * 100, digits = 2),
        ],
    );
    println("Paired excess (Bandit − Heuristic):")
    pretty_table(excess_df; backend = :text,
        fit_table_in_display_horizontally = false,
        fit_table_in_display_vertically = false,
        table_format = TextTableFormat(borders = text_table_borders__compact))

    # --- Panel 1: Paired-excess histogram ---
    p = histogram(excess ./ B₀, bins = 60, alpha = 0.7, color = :steelblue,
        xlabel = "Per-Path Excess (W_bandit - W_heuristic) / W₀",
        ylabel = "Count", title = "Bandit-η vs Heuristic-η: Paired Excess",
        legend = false, size = (800, 400))
    vline!(p, [0.0], lw = 2, c = :black, linestyle = :dash)
    vline!(p, [median(excess) / B₀], lw = 2, c = :coral, label = "")
    display(p)

    # --- Step 4: Learned eta map table ---
    eta_map = data["best_eta_per_regime"];
    eta_df = DataFrame(
        "Regime"     => ["bearish", "neutral", "bullish"],
        "Learned η"  => [eta_map[:bearish], eta_map[:neutral], eta_map[:bullish]],
    );
    println("Learned η map:")
    pretty_table(eta_df; backend = :text,
        fit_table_in_display_horizontally = false,
        fit_table_in_display_vertically = false,
        table_format = TextTableFormat(borders = text_table_borders__compact))
end;

___
## Summary
This example trained an epsilon-greedy elasticity bandit on a single path, applied the learned elasticity map across the Monte Carlo path ensemble alongside heuristic and fixed-elasticity baselines on the **same scenario with consistent bias treatment**, and compared the bandit and heuristic head-to-head. The learned elasticity map and bias-corrected backtest results are saved to `eta-bandit-results.jld2` for the validation report in Example 3.

> __Key Takeaways:__
>
> * __The bandit discovers regime-specific elasticity values:__ Different sentiment regimes may prefer different elasticity levels, and the bandit can break the symmetry that the heuristic imposes through its absolute-lambda dependence. Per-regime arm means make the preferred elasticity in each bin directly readable.
> * __Per-regime learning isolates the effect of elasticity choice:__ By holding preference weights and trigger rules fixed across the bandit, heuristic, and fixed-elasticity engines and applying the same bias-correction drag to all three, the backtest comparison isolates how much the elasticity-selection method contributes to performance. Cross-strategy variance in the bias-corrected scorecard is attributable to elasticity alone.
> * __The heuristic is a strong baseline:__ If the excess distribution centers near zero, the hand-designed formula was already close to optimal on this data. The bandit's value then lies in confirming that conclusion empirically rather than in discovering a new operating point.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The examples use synthetic data and simplified models.

___